# Chapter 5 · Measures of Frequency in Health: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only.

**Part A** follows the chapter's case, the annual report of the health department of a municipality of 200,000 inhabitants, through the department's questions; only the sections with a calculation appear (5.4, 5.9, 5.10, and 5.11). **Part B** gives the written solution of every chapter exercise (1 to 14; statements are in the book) and, for the exercises whose solution needs a calculation (13 and 14), its code and result. The code is Python 3 with pandas. Run the cells in order.

## Part A. The case

**Data.** `data/diabetes_cohort.csv`: 1,200 adults followed in primary care in 2025, one row per person (`person_id`, `entered`, `exited`, `diagnosed`, `exit_reason`). `data/mortality.csv`: population and deaths by age group (7 groups) in municipality A and in neighboring municipality B. `data/surveillance.csv`: 1,000 notified cases of an arbovirus infection (`case_id`, `month`, `age_group`, `district`, `hospitalized`, `died`), used by Exercise 13. The counts of the vaccination program and of the hospital are written in the cells. Each cell carries the number of the book section it belongs to.

### 5.4 Incidence

*How often do new cases of diabetes appear in the cohort?*

In [1]:
# 1. Read the follow-up table: one row per person, with the dates of entry and exit and whether the exit was a diagnosis
import pandas as pd

cohort = pd.read_csv("data/diabetes_cohort.csv", parse_dates=["entered", "exited"])

# 2. Follow-up of each person in whole days, and the person-years (the sum of the days divided by 365.25)
days = (cohort["exited"] - cohort["entered"]).dt.days
person_years = days.sum() / 365.25
new_cases = cohort["diagnosed"].sum()
print("Persons:", len(cohort), "; new cases:", new_cases, "; person-years:", round(person_years, 1))

# 3. The observed diagnosis fraction (cases divided by persons) and the incidence rate (cases divided by person-time)
print("Observed diagnosis fraction:", round(100 * new_cases / len(cohort), 2), "%")
print("Incidence rate:", round(1000 * new_cases / person_years, 1), "per 1,000 person-years")

Persons: 1200 ; new cases: 31 ; person-years: 1135.8
Observed diagnosis fraction: 2.58 %
Incidence rate: 27.3 per 1,000 person-years


### 5.9 Program coverage

*Was DTP vaccine coverage really 104%?*

In [2]:
# 1. The counts of the program, each kept with its own denominator
doses = 2496
cohort_size = 2400
outside = 138
duplicated = 61
children_3 = 600
valid = doses - outside - duplicated

# 2. The administrative estimate: doses divided by children
print("Administrative coverage (doses over children):", round(100 * doses / cohort_size, 1), "%")

# 3. The audited dose coverage: valid doses over the three doses that each child needs
print("Audited dose coverage (valid doses over 3 doses per child):", round(100 * valid / (3 * cohort_size), 1), "%")

# 4. The proportion of children with three doses
print("Children with three doses:", round(100 * children_3 / cohort_size, 1), "%")

# 5. The survey estimate: 98 vaccinated among 420 children
print("Survey estimate:", round(100 * 98 / 420, 1), "%")

Administrative coverage (doses over children): 104.0 %
Audited dose coverage (valid doses over 3 doses per child): 31.9 %
Children with three doses: 25.0 %
Survey estimate: 23.3 %


### 5.10 Crude and standardized rates

*Is mortality really higher in the neighboring municipality?*

In [3]:
# 1. Read the deaths and the populations by age group of towns A and B
import pandas as pd

m = pd.read_csv("data/mortality.csv")

# 2. Crude rates per 1,000: all deaths over the whole population
print("Crude rate, town A:", round(1000 * m["deaths_a"].sum() / m["population_a"].sum(), 2), "; town B:", round(1000 * m["deaths_b"].sum() / m["population_b"].sum(), 2))

# 3. Direct standardization: the age-specific rates of each town applied to the combined population of A and B (the standard)
rate_a = m["deaths_a"] / m["population_a"]
rate_b = m["deaths_b"] / m["population_b"]
standard = m["population_a"] + m["population_b"]
print("Standardized rate, town A:", round(1000 * (rate_a * standard).sum() / standard.sum(), 2), "; town B:", round(1000 * (rate_b * standard).sum() / standard.sum(), 2))

# 4. Indirect standardization: deaths expected in B under the age-specific rates of A, and the ratio of observed to expected deaths
expected_b = (rate_a * m["population_b"]).sum()
print("Observed deaths in B:", m["deaths_b"].sum(), "; expected:", round(expected_b))
print("Standardized mortality ratio of B (observed over expected):", round(m["deaths_b"].sum() / expected_b, 3))

Crude rate, town A: 7.56 ; town B: 12.39
Standardized rate, town A: 10.39 ; town B: 9.4
Observed deaths in B: 1859 ; expected: 2124
Standardized mortality ratio of B (observed over expected): 0.875


### 5.11 Hospital service indicators

*How are the beds of the 300-bed hospital used?*

In [4]:
# 1. The year of a 300-bed hospital: beds, days, occupied bed-days, and stays that ended
beds = 300
days = 365
occupied_bed_days = 91980
ended_stays = 12264

# 2. The three indicators
occupancy = occupied_bed_days / (beds * days)
period_stay = occupied_bed_days / ended_stays
turnover = ended_stays / beds
print("Occupancy:", round(100 * occupancy, 1), "%")
print("Period stay indicator:", round(period_stay, 1), "days")
print("Turnover:", round(turnover, 1), "stays per bed")

# 3. Turnover from the identity: days times occupancy divided by the period stay indicator
print("Turnover from the identity:", round(days * occupancy / period_stay, 1), "stays per bed")

Occupancy: 84.0 %
Period stay indicator: 7.5 days
Turnover: 40.9 stays per bed
Turnover from the identity: 40.9 stays per bed


## Part B. Exercise solutions

### 5.14 Exercise 1. Three measures, three denominators

Residents aged 80 or more: $6{,}000/120{,}000=5\%$, a population proportion.
Nursing-home residents: $120/6{,}000=2\%$ of the older group, another proportion.
Annual influenza mortality in that group is $42/6{,}000\times100{,}000=700$
per 100,000 person-years, using its population as an approximation to annual
person-time. The third measure estimates an event rate; its time denominator
is implicit in the annual population calculation.

### 5.14 Exercise 2. A screening result

$84/2{,}800=3\%$ is a prevalence of previously undetected disease at the time of
screening: it counts existing cases, of any duration, among the women examined.
It says nothing about how many women develop the condition each year, which
would be an incidence and would require following women free of the condition
over time. A screening program detects at its first round the cases that have
accumulated over many years.

### 5.14 Exercise 3. Cumulative incidence and rate

The observed diagnosis fraction is $18/950=1.89\%$. It is not the fully
observed one-year risk, since 40 nurses have unknown outcomes after leaving.
The risk is between $18/950=1.89\%$ and $(18+40)/950=6.11\%$ if the recorded
diagnoses are correct and those are the only missing outcomes.
For person-time, 892 nurses contribute a full year and the 58 who left or
developed pain contribute half a year each: $892+29=921$ person-years.
The incidence rate is $18/921\times1{,}000=19.5$ per 1,000 person-years.
The percentage and rate have different units; one is not simply a larger
version of the other.

### 5.14 Exercise 4. Prevalence from incidence and duration

For X, $ID=0.032\times8=0.256$, whereas $ID/(1+ID)=0.256/1.256=20.4\%$.
For Y, $ID=0.004\times4=0.016$, whereas $0.016/1.016=1.57\%$.
The rare-condition approximation gives 25.6% and 1.6%, respectively; it is
much less accurate for X. Under the stated balance model, X affects about
13 times as many people at a moment and arises eight times as quickly among
people free of the condition. Prevalence describes the potential volume of
continuing care, but service needs also depend on illness severity and care
requirements.

### 5.14 Exercise 5. A wedding dinner

Attack rate: $58/240=24.2\%$. Proportion of cases admitted: $12/58=20.7\%$, a
measure of severity among cases. Attack rate among those who ate the starter:
$27/95=28.4\%$. The last figure helps identify the cause only when compared
with the attack rate among those who did not eat the starter,
$(58-27)/(240-95)=31/145=21.4\%$; that comparison is a measure of association,
developed in Chapter 7.

### 5.14 Exercise 6. Three denominators in one town

Case fatality: $4/100=4\%$, deaths over cases; the severity of the disease.
Cause-specific mortality: $(4/3{,}000)\times100{,}000\approx133$ per 100,000, deaths over population;
the annual rate of deaths attributed to it in the town. Proportional mortality: $4/26=15.4\%$ of all
deaths; the share of the town's dying attributable to the disease, useful for
priorities but not a risk.

### 5.14 Exercise 7. Infant and neonatal mortality

$3.8/18.2=20.9\%$ of infant deaths occurred before 28 completed days. The neonatal
deaths are part of the infant deaths: the numerator of the neonatal rate is
contained in the numerator of the infant rate, and both share the denominator
of live births.

### 5.14 Exercise 8. Case fatality during an epidemic

The division gives 7.4%, but the statement does not tell us how many cases
have completed a common follow-up window. If some outcomes are still pending,
additional deaths can raise the final proportion among these same 1,000
cases. The 27 recent deaths do not themselves prove how many outcomes are
unresolved. Follow a defined case cohort to the specified endpoint and verify
that all deaths belong to those cases. Preferential detection of severe disease
can make case fatality exceed infection fatality, because mild and asymptomatic
infections are missing from the case denominator. The magnitude, and whether
both biases are present, cannot be determined from the supplied counts alone.

### 5.14 Exercise 9. Three coverages

Administrative: $1{,}880/1{,}750=107.4\%$. Corrected:
$(1{,}880-120-45)/1{,}750=98.0\%$. Survey: $322/360=89.4\%$. A suitable
paragraph: “Administrative coverage of the first dose was 107.4% (1,880
doses for an estimated cohort of 1,750). After removing 120 doses given to
non-residents and 45 duplicate records, coverage was 98.0%. A household survey
of 360 infants found 89.4% vaccinated. The register and the population estimate
are being reviewed.”

### 5.14 Exercise 10. Standardizing two districts

Crude rates: A, $140/50{,}000=2.80$ per 1,000; B, $298/50{,}000=5.96$.
Age-specific rates per 1,000: A, 1.00 under 65 and 10.0 at 65 and over; B,
1.40 and 9.00. Standard population: 60,000 under 65 and 40,000 at 65 and
over. Standardized rates: A, $(1.00\times60{,}000+10.0\times40{,}000)/100{,}000=4.60$;
B, $(1.40\times60{,}000+9.00\times40{,}000)/100{,}000=4.44$. Expected deaths in B
at A's rates: $0.001\times20{,}000+0.010\times30{,}000=320$; $SMR=298/320=0.93$.
After standardization A has the slightly higher mortality; B's higher crude rate
reflects its older population.

### 5.14 Exercise 11. A 150-bed hospital

Occupancy: $51{,}100/(150\times365)\approx93.3\%$. Period stay indicator:
$51{,}100/6{,}570\approx7.78$ days. Turnover: $6{,}570/150=43.8$ stays per bed.
Using unrounded quantities verifies the identity exactly. These aggregates do
not supply the mean full duration of stays discharged in 2025. In 2026 the closed ward removes
$30\times120=3{,}600$ bed-days from the denominator of occupancy, which becomes
$150\times365-3{,}600=51{,}150$ available bed-days.

### 5.14 Exercise 12. Odds and proportions

$0.15/0.85=0.176$. Odds of 1.5 give a proportion of $1.5/2.5=0.60$. Odds of 3
give $3/4=0.75$: three quarters of patients were readmitted. Odds exceed the
proportion by the factor $1/(1-p)$, which is below 1.05 when $p<1/21\approx0.04762$. For $p>0$, the relative difference is
$[p/(1-p)-p]/p=p/(1-p)$, so the threshold is about 4.76%, not exactly 5%.

### 5.14 Exercise 13. Incidence by district

North: 552 cases among 110,000, 501.8 per 100,000; south: 448 among
90,000, 497.8 per 100,000. The incidences are practically equal, although
the north has more cases. Case fatality among hospitalized cases: north 2 of 51,
3.9%; south 1 of 42, 2.4%. These are small numbers, and the difference cannot
be interpreted without the methods of later chapters.

In [5]:
# 1. Incidence per 100,000 by district: the cases of each district divided by its own population
cases = pd.read_csv("data/surveillance.csv")
population = {"north": 110000, "south": 90000}
by_district = cases.groupby("district").size()
print("Incidence per 100,000 by district:", (1e5 * by_district / pd.Series(population)).round(1).to_dict())

# 2. Case fatality (%) among the hospitalized cases of each district
admitted = cases[cases["hospitalized"]].groupby("district")
print("Case fatality among hospitalized cases, % by district:", (100 * admitted["died"].sum() / admitted.size()).round(1).to_dict())

Incidence per 100,000 by district: {'north': 501.8, 'south': 497.8}
Case fatality among hospitalized cases, % by district: {'north': 3.9, 'south': 2.4}


### 5.14 Exercise 14. A different standard

With A's own population as the standard, A's standardized rate equals its
crude rate, 7.56 per 1,000, and B's becomes 7.16. The ordering is unchanged, A
higher than B, but both numbers differ from those obtained with the combined
standard, 10.39 and 9.40. A standardized rate is a device for comparison under
one stated standard; it describes the hypothetical rate under the stated age distribution.
The present ranking is unchanged, but a different standard can change a ranking
when the age-specific rate curves cross.

In [6]:
# 1. Standardization with the population of town A as the standard
standard = m["population_a"]
rate_a = m["deaths_a"] / m["population_a"]
rate_b = m["deaths_b"] / m["population_b"]
print("Standardized rate with A as the standard - town A:", round(1000 * (rate_a * standard).sum() / standard.sum(), 2), "; town B:", round(1000 * (rate_b * standard).sum() / standard.sum(), 2))

Standardized rate with A as the standard - town A: 7.56 ; town B: 7.16


---
*Synthetic data (`generate_data.py`, seeds 5 and 51). Teaching values only; nothing here is an estimate for a real municipality.*